In [0]:
gold_table_candidates = ["gold_battery_health_summary", "gold_daily_battery_kpis"]
gold_path_candidates = ["abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/gold/daily_battery_kpis"]

df_gold = None
last_error = None

for t in gold_table_candidates:
    try:
        df_gold = spark.read.table(t)
        print(f"Loaded Gold table: {t}")
        break
    except Exception as e:
        last_error = e

if df_gold is None:
    for p in gold_path_candidates:
        try:
            df_gold = spark.read.format("delta").load(p)
            print(f"Loaded Gold path: {p}")
            break
        except Exception as e:
            last_error = e

if df_gold is None:
    raise last_error
display(df_gold.limit(20))

In [0]:
from pyspark.sql import functions as F

share_base = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/shared/exports"
csv_path = f"{share_base}/daily_battery_kpis_csv"
json_path = f"{share_base}/latest_battery_summary_json"
stats_path = f"{share_base}/pipeline_statistics"

(df_gold
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", "true")
 .csv(csv_path))

latest_col = "date" if "date" in df_gold.columns else df_gold.columns[0]
(df_gold
 .orderBy(F.col(latest_col).desc())
 .limit(50)
 .coalesce(1)
 .write
 .mode("overwrite")
 .json(json_path))

stats_df = spark.createDataFrame([{
    "dataset_name": "gold_daily_battery_kpis",
    "record_count": df_gold.count(),
    "column_count": len(df_gold.columns),
    "export_csv_path": csv_path,
    "export_json_path": json_path
}])

(stats_df
 .write
 .format("delta")
 .mode("overwrite")
 .save(stats_path))

print("CSV export:", csv_path)
print("JSON export:", json_path)
print("Stats path:", stats_path)

In [0]:
display(spark.read.format("delta").load(stats_path))